# Bundesliga 2023/2024: Tracking Data Extraction

#### 1. Import Required Libraries

In [1]:
import duckdb
import pandas as pd
import numpy as np
import os

#### 2. Load and Explore the Dataset


In [2]:
# Connection to DuckDB
conn = duckdb.connect('Bundesliga_2023_2024.duckdb')

In [3]:
# Helper to inspect DB schema 
def show_database_schema(conn):
    """Displays the complete schema of all tables in the connected DuckDB database."""
    tables = conn.execute("SHOW TABLES").fetchall()
    
    for table_name in tables:
        table = table_name[0]
        print(f"\nTable: {table}")
        print("-" * 60)
        schema = conn.execute(f"DESCRIBE {table}").df()
        print(schema)

# Display the database structure
show_database_schema(conn)


Table: match_information
------------------------------------------------------------
            column_name column_type null   key default extra
0                    id     VARCHAR  YES  None    None  None
1           match_title     VARCHAR  YES  None    None  None
2              matchday     INTEGER  YES  None    None  None
3  planned_kickoff_time   TIMESTAMP  YES  None    None  None
4          home_team_id     VARCHAR  YES  None    None  None
5          away_team_id     VARCHAR  YES  None    None  None
6                result     VARCHAR  YES  None    None  None

Table: players
------------------------------------------------------------
     column_name column_type null   key default extra
0             id     VARCHAR  YES  None    None  None
1           name     VARCHAR  YES  None    None  None
2     first_name     VARCHAR  YES  None    None  None
3      last_name     VARCHAR  YES  None    None  None
4     short_name     VARCHAR  YES  None    None  None
5      birthdate        

In [4]:
# Create (if not existing) a directory for CSV exports
export_dir = "exports_per_day"
os.makedirs(export_dir, exist_ok=True)

In [5]:
# Query basic match information (IDs, titles, matchdays, results)
query = """
SELECT DISTINCT 
    tr.match_id, 
    mi.match_title, 
    mi.matchday, 
    mi.result
FROM tracking_raw_observed AS tr
LEFT JOIN match_information mi ON mi.id = tr.match_id
ORDER BY mi.matchday;
"""

# Execute the query and fetch results as a pandas DataFrame
match_df = conn.execute(query).fetchdf()

# Convert matchday values to Python integers for sorting and iteration
matchdays = sorted(int(day) for day in match_df.matchday.unique())
print(f"Detected matchdays: {matchdays}")


Detected matchdays: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34]


#### 3. Loop Through Matchdays and Export Data

In [1]:
# For each selected matchday, fetch player tracking data,
# clean it, and export the result to a separate CSV file.

for day in matchdays:
    print(f"\nProcessing matchday {day}...")

    # SQL query: retrieve all player positions and related information for the given matchday
    query = f"""
    SELECT 
        m.match_title,
        tr.match_id,
        t.three_letter_code AS team_code,
        home.three_letter_code AS home_team_code,
        away.three_letter_code AS away_team_code,
        p.short_name AS player_name,
        tr.frame,
        tr.game_section,
        tr.x_position,
        tr.y_position,
        tr.z_position,
        tr.speed,
        tr.ball_status,
        tr.ball_possession
    FROM tracking_raw_observed tr
    LEFT JOIN match_information m ON tr.match_id = m.id
    LEFT JOIN teams t ON tr.team_id = t.id
    LEFT JOIN teams AS home ON m.home_team_id = home.id
    LEFT JOIN teams AS away ON m.away_team_id = away.id
    LEFT JOIN players p ON tr.player_id = p.id
    WHERE m.matchday = {day}
    ORDER BY tr.match_id, tr.frame, team_code, x_position;
    """

    # Execute query and fetch as DataFrame
    pos = conn.execute(query).fetchdf()

    # Identify all frames where the ball was not in play (ball_status == False)
    # These will be removed from the dataset
    frames_to_drop = pos.loc[pos['ball_status'] == False, ['match_title', 'frame']]

    # Perform an anti-join: remove rows corresponding to those invalid frames
    pos_cleaned = pos.merge(
        frames_to_drop,
        on=['match_title', 'frame'],
        how='left',
        indicator=True
    )
    pos_cleaned = pos_cleaned[pos_cleaned['_merge'] == 'left_only'].drop(columns='_merge')

    # Define output path and export cleaned data to CSV
    output_path = os.path.join(export_dir, f"tracking_data_day_{day:02d}.csv")
    pos_cleaned.to_csv(output_path, index=False)

    print(f"Saved {output_path} ({len(pos_cleaned):,} rows after cleaning)")


Processing matchday 1...
Saved /home/falih/Mastercode/Bundesliga_2023_2024/exports_per_day/tracking_data_day_01.csv (18,071,169 rows after cleaning)
Processing matchday 2...
Saved /home/falih/Mastercode/Bundesliga_2023_2024/exports_per_day/tracking_data_day_02.csv (17,548,549 rows after cleaning)
Processing matchday 3...
Saved /home/falih/Mastercode/Bundesliga_2023_2024/exports_per_day/tracking_data_day_03.csv (17,809,088 rows after cleaning)
Processing matchday 4...
Saved /home/falih/Mastercode/Bundesliga_2023_2024/exports_per_day/tracking_data_day_04.csv (17,741,496 rows after cleaning)
Processing matchday 5...
Saved /home/falih/Mastercode/Bundesliga_2023_2024/exports_per_day/tracking_data_day_05.csv (18,501,497 rows after cleaning)
Processing matchday 6...
Saved /home/falih/Mastercode/Bundesliga_2023_2024/exports_per_day/tracking_data_day_06.csv (17,395,710 rows after cleaning)
Processing matchday 7...
Saved /home/falih/Mastercode/Bundesliga_2023_2024/exports_per_day/tracking_data_d

In [7]:
# Close database connection
conn.close()